In [ ]:
# benchmark_resnet50_inference.py
import torch
import torch.nn as nn
from torchvision import models, transforms
import time
import numpy as np
import onnx
import onnxruntime as ort
from tqdm import tqdm
import os

In [ ]:

# Конфигурация
BATCH_SIZES = [1, 32, 128, 256]
NUM_WARMUP_ITERATIONS = 5
NUM_BENCHMARK_ITERATIONS = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Устройство: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA версия: {torch.version.cuda}")

# Функция загрузки модели
def load_model(model_path=None, num_classes=10):
    """Загрузка модели ResNet-50 для CIFAR-10"""
    model = models.resnet50(pretrained=False, num_classes=num_classes)
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        print(f"Загружены веса из {model_path}")
    else:
        print("Веса не найдены, используется случайная инициализация")
    model = model.to(DEVICE)
    model.eval()
    return model

# Точный таймер для GPU
class CUDATimer:
    def __init__(self):
        self.reset()
    
    def reset(self):
        if DEVICE.type == "cuda":
            self.start_event = torch.cuda.Event(enable_timing=True)
            self.end_event = torch.cuda.Event(enable_timing=True)
            self.start_event.record()
        else:
            self.start_time = time.time()
    
    def elapsed(self):
        if DEVICE.type == "cuda":
            self.end_event.record()
            torch.cuda.synchronize()
            return self.start_event.elapsed_time(self.end_event) / 1000.0
        return time.time() - self.start_time

# Функция бенчмарка инференса
def benchmark_inference(model, input_shape=(3, 224, 224), model_name="Model"):
    """Замер скорости инференса для разных batch sizes"""
    print(f"\n{'='*60}")
    print(f"Бенчмарк: {model_name}")
    print(f"{'='*60}")
    
    results = {}
    
    for batch_size in BATCH_SIZES:
        # Создаем случайный вход
        dummy_input = torch.randn(batch_size, *input_shape).to(DEVICE)
        
        # Прогрев (warmup)
        with torch.no_grad():
            for _ in range(NUM_WARMUP_ITERATIONS):
                _ = model(dummy_input)
        
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        
        # Замер
        timer = CUDATimer()
        with torch.no_grad():
            for _ in range(NUM_BENCHMARK_ITERATIONS):
                _ = model(dummy_input)
        
        elapsed_time = timer.elapsed()
        fps = (NUM_BENCHMARK_ITERATIONS * batch_size) / elapsed_time
        latency_ms = (elapsed_time / NUM_BENCHMARK_ITERATIONS) * 1000
        
        results[batch_size] = {
            'fps': fps,
            'latency_ms': latency_ms,
            'time_seconds': elapsed_time
        }
        
        print(f"  Batch Size {batch_size:3d}: {fps:7.2f} img/sec, "
              f"Latency: {latency_ms:5.2f} ms/batch")
    
    return results

# Визуализация сравнения
def compare_results(all_results):
    """Сравнение всех методов ускорения"""
    print(f"\n{'='*80}")
    print("СРАВНЕНИЕ МЕТОДОВ УСКОРЕНИЯ")
    print(f"{'='*80}")
    
    # Заголовок таблицы
    print(f"\n{'Batch Size':<12}", end="")
    for method in all_results.keys():
        print(f"{method:>15}", end="")
    print()
    
    # Для каждого batch size
    for bs in BATCH_SIZES:
        print(f"{bs:<12}", end="")
        for method, results in all_results.items():
            if bs in results:
                print(f"{results[bs]['fps']:>14.1f}", end="")
            else:
                print(f"{'N/A':>14}", end="")
        print()
    
    # Относительное ускорение относительно baseline
    if 'Baseline (PyTorch)' in all_results:
        print(f"\n{'='*80}")
        print("ОТНОСИТЕЛЬНОЕ УСКОРЕНИЕ (Baseline = 1.0x)")
        print(f"{'='*80}")
        
        print(f"\n{'Batch Size':<12}", end="")
        for method in all_results.keys():
            if method != 'Baseline (PyTorch)':
                print(f"{method:>15}", end="")
        print()
        
        baseline = all_results['Baseline (PyTorch)']
        for bs in BATCH_SIZES:
            if bs in baseline:
                print(f"{bs:<12}", end="")
                for method, results in all_results.items():
                    if method != 'Baseline (PyTorch)' and bs in results:
                        speedup = results[bs]['fps'] / baseline[bs]['fps']
                        print(f"{speedup:>14.2f}x", end="")
                print()

# ============================================
# 1. BASELINE: Обычный PyTorch Eager
# ============================================
def benchmark_baseline(model):
    """Базовый PyTorch без оптимизаций"""
    return benchmark_inference(model, model_name="Baseline (PyTorch)")

# ============================================
# 2. TORCH.COMPILE (JIT компиляция)
# ============================================
def benchmark_torch_compile(model):
    """Использование torch.compile с разными режимами"""
    results = {}
    
    for mode in ['default', 'reduce-overhead', 'max-autotune']:
        print(f"\n  Компиляция с режимом '{mode}'...")
        try:
            compiled_model = torch.compile(model, mode=mode)
            # Прогрев для компиляции
            dummy = torch.randn(32, 3, 224, 224).to(DEVICE)
            for _ in range(10):
                _ = compiled_model(dummy)
            
            res = benchmark_inference(compiled_model, model_name=f"torch.compile ({mode})")
            results[mode] = res
        except Exception as e:
            print(f"  Ошибка с режимом {mode}: {e}")
    
    return results

# ============================================
# 3. MIXED PRECISION (AMP)
# ============================================
def benchmark_mixed_precision(model):
    """Automatic Mixed Precision (FP16/BF16)"""
    
    # Создаем обертку для AMP
    class AMPWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
            self.dtype = torch.float16 if DEVICE.type == "cuda" else torch.bfloat16
        
        @torch.no_grad()
        def forward(self, x):
            with torch.cuda.amp.autocast(dtype=self.dtype):
                return self.model(x)
    
    amp_model = AMPWrapper(model).to(DEVICE)
    return benchmark_inference(amp_model, model_name="Mixed Precision (AMP FP16)")

# ============================================
# 4. ONNX RUNTIME (без TensorRT)
# ============================================
def benchmark_onnx_runtime(model):
    """Экспорт в ONNX и инференс через ONNX Runtime"""
    
    # Экспорт в ONNX
    onnx_path = "model.onnx"
    dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
    
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        },
        opset_version=14
    )
    
    # Загрузка ONNX модели
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    
    # Инференс через ONNX Runtime
    ort_session = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    
    # Бенчмарк
    results = {}
    for batch_size in BATCH_SIZES:
        dummy_input_np = np.random.randn(batch_size, 3, 224, 224).astype(np.float32)
        
        # Прогрев
        for _ in range(10):
            ort_session.run(None, {'input': dummy_input_np})
        
        # Замер
        timer = time.time()
        for _ in range(NUM_BENCHMARK_ITERATIONS):
            ort_session.run(None, {'input': dummy_input_np})
        elapsed = time.time() - timer
        
        fps = (NUM_BENCHMARK_ITERATIONS * batch_size) / elapsed
        results[batch_size] = {'fps': fps, 'latency_ms': (elapsed/NUM_BENCHMARK_ITERATIONS)*1000}
        print(f"  Batch Size {batch_size:3d}: {fps:7.2f} img/sec")
    
    # Очистка
    os.remove(onnx_path)
    return results

# ============================================
# 5. ONNX → TENSORRT
# ============================================
def benchmark_onnx_tensorrt(model):
    """Конвертация ONNX → TensorRT Engine"""
    
    try:
        import tensorrt as trt
        from torch2trt import torch2trt
    except ImportError:
        print("  TensorRT или torch2trt не установлены")
        return None
    
    # Экспорт в ONNX
    onnx_path = "model.onnx"
    dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
    
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=['input'],
        output_names=['output'],
        opset_version=14
    )
    
    # Конвертация в TensorRT
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    
    with open(onnx_path, 'rb') as f:
        parser.parse(f.read())
    
    # Конфигурация
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB
    
    # Оптимизация для разных batch sizes
    profile = builder.create_optimization_profile()
    for bs in BATCH_SIZES:
        profile.set_shape('input', (bs, 3, 224, 224), (bs, 3, 224, 224), (bs, 3, 224, 224))
    config.add_optimization_profile(profile)
    
    # Сборка engine
    engine = builder.build_serialized_network(network, config)
    runtime = trt.Runtime(logger)
    trt_engine = runtime.deserialize_cuda_engine(engine)
    
    # Бенчмарк
    results = {}
    context = trt_engine.create_execution_context()
    
    for batch_size in BATCH_SIZES:
        # Подготовка буферов
        input_buf = torch.randn(batch_size, 3, 224, 224).cuda()
        output_buf = torch.empty(batch_size, 10).cuda()
        
        # Прогрев
        for _ in range(10):
            context.execute_v2([input_buf.data_ptr(), output_buf.data_ptr()])
        
        torch.cuda.synchronize()
        
        # Замер
        timer = CUDATimer()
        for _ in range(NUM_BENCHMARK_ITERATIONS):
            context.execute_v2([input_buf.data_ptr(), output_buf.data_ptr()])
        torch.cuda.synchronize()
        
        elapsed = timer.elapsed()
        fps = (NUM_BENCHMARK_ITERATIONS * batch_size) / elapsed
        results[batch_size] = {'fps': fps, 'latency_ms': (elapsed/NUM_BENCHMARK_ITERATIONS)*1000}
        print(f"  Batch Size {batch_size:3d}: {fps:7.2f} img/sec")
    
    # Очистка
    os.remove(onnx_path)
    return results

# ============================================
# 6. TORCH-TENSORRT (гибридный подход)
# ============================================
def benchmark_torch_tensorrt(model):
    """Использование torch_tensorrt напрямую"""
    
    try:
        import torch_tensorrt
    except ImportError:
        print("  torch_tensorrt не установлен")
        return None
    
    # Конвертация
    dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
    
    compile_spec = {
        "inputs": [dummy_input],
        "enabled_precisions": {torch.float16},  # Используем FP16
        "workspace_size": 1 << 30  # 1GB
    }
    
    trt_model = torch_tensorrt.compile(model, **compile_spec)
    
    return benchmark_inference(trt_model, model_name="torch_tensorrt (FP16)")

# ============================================
# ОСНОВНОЙ ПАЙПЛАЙН
# ============================================   

Устройство: cuda
GPU: Tesla V100-SXM2-16GB
CUDA версия: 11.7


In [ ]:
print("\n" + "="*80)
print("БЕНЧМАРК МЕТОДОВ УСКОРЕНИЯ RESNET-50 НА CIFAR-10")
print("="*80)
    
# Загрузка модели
model_path = "resnet50_cifar10_baseline.pth"
model = load_model(model_path, num_classes=10)
    
all_results = {}
    
# 1. Baseline
all_results['Baseline (PyTorch)'] = benchmark_baseline(model)


БЕНЧМАРК МЕТОДОВ УСКОРЕНИЯ RESNET-50 НА CIFAR-10
Загружены веса из resnet50_cifar10_baseline.pth

Бенчмарк: Baseline (PyTorch)


/home/tachkin/miniconda3/envs/torch_v100_pip/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/tachkin/miniconda3/envs/torch_v100_pip/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  Batch Size   1:  363.93 img/sec, Latency:  2.75 ms/batch
  Batch Size  32: 1158.05 img/sec, Latency: 27.63 ms/batch
  Batch Size 128: 1251.21 img/sec, Latency: 102.30 ms/batch


OutOfMemoryError: CUDA out of memory. Tried to allocate 784.00 MiB (GPU 0; 15.77 GiB total capacity; 2.05 GiB already allocated; 9.50 MiB free; 2.81 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
# 2. torch.compile
print("\n" + "="*60)
print("Метод 1: torch.compile")
print("="*60)
compile_results = benchmark_torch_compile(model)
for mode, res in compile_results.items():
    all_results[f"torch.compile ({mode})"] = res

In [ ]:

    
# 3. Mixed Precision
print("\n" + "="*60)
print("Метод 2: Mixed Precision (AMP)")
print("="*60)
all_results['Mixed Precision (AMP)'] = benchmark_mixed_precision(model)
    
# 4. ONNX Runtime
print("\n" + "="*60)
print("Метод 3: ONNX Runtime")
print("="*60)
all_results['ONNX Runtime'] = benchmark_onnx_runtime(model)
    
# 5. ONNX → TensorRT (только если есть TensorRT)
print("\n" + "="*60)
print("Метод 4: ONNX → TensorRT")
print("="*60)
trt_results = benchmark_onnx_tensorrt(model)
if trt_results:
    all_results['ONNX → TensorRT'] = trt_results
    
# 6. torch_tensorrt (только если есть torch_tensorrt)
print("\n" + "="*60)
print("Метод 5: torch_tensorrt")
print("="*60)
torch_trt_results = benchmark_torch_tensorrt(model)
if torch_trt_results:
    all_results['torch_tensorrt'] = torch_trt_results
    
# Сравнение всех методов
compare_results(all_results)
    
print("\n" + "="*80)
print("БЕНЧМАРК ЗАВЕРШЕН")
print("="*80)